In [ ]:
# Get pipeline settings from bundle configuration
bronze_catalog = spark.conf.get("bronze_catalog")
bronze_schema = spark.conf.get("bronze_schema")
silver_catalog = spark.conf.get("silver_catalog")
silver_schema = spark.conf.get("silver_schema")
gold_catalog = spark.conf.get("gold_catalog")
fact_schema = spark.conf.get("fact_schema")
dim_schema = spark.conf.get("dim_schema")

In [ ]:
from pyspark import pipelines as dp

## Bronze Layer

Pull in raw data from our source system with minimal transformation.

In [ ]:
# Pull in taxi data
@dp.table(name=f"{bronze_catalog}.{bronze_schema}.yellow_taxi_trips"
          , comment="Raw yellow taxi trip data from NYC Taxi and Limousine Commission (TLC)"
          , table_properties={
              "quality": "bronze"
                , "delta.autoOptimize.autoCompact":"true"
                , "delta.autoOptimize.optimizeWrite":"true"
                , "delta.enableRowTracking":"true"
                , "delta.enableDeletionVectors":"true"
            }
)
def yellow_taxi_trips():
    return spark.table("samples.nyctaxi.trips")

## Silver Layer

Transform raw data into a more refined format for analysis; dropping invalid records.

In [ ]:
@dp.table(name=f"{silver_catalog}.{silver_schema}.yellow_taxi_trips_cleaned"
          , comment="Cleaned yellow taxi trip data from NYC Taxi and Limousine Commission (TLC)"
          , table_properties={
              "quality": "silver"
                , "delta.autoOptimize.autoCompact":"true"
                , "delta.autoOptimize.optimizeWrite":"true"
                , "delta.enableRowTracking":"true"
                , "delta.enableDeletionVectors":"true"
            }
)
@dp.expect_all_or_drop(
    {
        "valid_dropoff_time": "tpep_dropoff_datetime >= tpep_pickup_datetime"
        , "valid_trip_distance": "trip_distance > 0"
    }
)
def yellow_taxi_trips_cleaned():
    return spark.table(f"{bronze_catalog}.{bronze_schema}.yellow_taxi_trips")

## Gold Layer

Create business-level aggregates and metrics for reporting and analytics.

In [ ]:
@dp.table(name=f"{gold_catalog}.{fact_schema}.yellow_taxi_trips_cleaned"
           , comment="Fact table for yellow taxi trip data from NYC Taxi and Limousine Commission (TLC)"
           , table_properties={
               "quality": "gold"
                , "delta.autoOptimize.autoCompact":"true"
                , "delta.autoOptimize.optimizeWrite":"true"
                , "delta.enableRowTracking":"true"
                , "delta.enableDeletionVectors":"true"
            }
)
def fact_yellow_taxi_trips():
    return spark.sql(f"""
                    SELECT
                     date(tpep_pickup_datetime) as trip_date
                     ,pickup_zip
                     ,count(*) as total_trips
                     ,sum(fare_amount) as total_fare_amount
                     ,avg(trip_distance) as avg_trip_distance
                     ,avg(date_diff(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime)) as avg_total_trip_minutes
                     from {silver_catalog}.{silver_schema}.yellow_taxi_trips_cleaned
                     group by all;
                     """)

In [ ]:
@dp.materialized_view(name=f"{gold_catalog}.{fact_schema}.pipeline_observability_metrics",
                      comment="Daily data quality and throughput metrics for demo dashboards")
def pipeline_observability_metrics():
    return spark.sql(f"""
        WITH bronze_daily AS (
            SELECT
                date(tpep_pickup_datetime) AS metric_date,
                count(*) AS bronze_rows
            FROM {bronze_catalog}.{bronze_schema}.yellow_taxi_trips
            GROUP BY 1
        ),
        silver_daily AS (
            SELECT
                date(tpep_pickup_datetime) AS metric_date,
                count(*) AS silver_rows
            FROM {silver_catalog}.{silver_schema}.yellow_taxi_trips_cleaned
            GROUP BY 1
        ),
        fact_daily AS (
            SELECT
                trip_date AS metric_date,
                avg(avg_total_trip_minutes) AS avg_trip_minutes
            FROM {gold_catalog}.{fact_schema}.yellow_taxi_trips_cleaned
            GROUP BY 1
        )
        SELECT
            coalesce(b.metric_date, s.metric_date) AS metric_date,
            coalesce(b.bronze_rows, 0) AS rows_loaded,
            coalesce(s.silver_rows, 0) AS rows_after_quality_filters,
            greatest(coalesce(b.bronze_rows, 0) - coalesce(s.silver_rows, 0), 0) AS rows_quarantined,
            CASE
                WHEN coalesce(b.bronze_rows, 0) = 0 THEN 0.0
                ELSE round((greatest(coalesce(b.bronze_rows, 0) - coalesce(s.silver_rows, 0), 0) * 100.0) / coalesce(b.bronze_rows, 0), 2)
            END AS quarantine_rate_pct,
            f.avg_trip_minutes
        FROM bronze_daily b
        FULL OUTER JOIN silver_daily s
            ON b.metric_date = s.metric_date
        LEFT JOIN fact_daily f
            ON coalesce(b.metric_date, s.metric_date) = f.metric_date
    """)